# 04 - Hybrid Ensemble
Tunes and trains the final XGBoost stacker over features + GraphSAGE embeddings.

In [ ]:
from pathlib import Path
import os
import sys
import joblib
import mlflow
import optuna
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
features = pd.read_parquet('data/processed/features.parquet')
emb = pd.read_parquet('data/processed/gnn_embeddings.parquet')
df = features.merge(emb, on='borrower_id', how='inner')
x = df.drop(columns=['borrower_id', 'default_flag'], errors='ignore')
y = df['default_flag'].astype(int)

x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.30, random_state=42, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 700),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 5.0),
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1,
    }
    model = XGBClassifier(**params)
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    score = model.predict_proba(x_val)[:, 1]
    return roc_auc_score(y_val, score)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)
study.best_params

In [ ]:
best = study.best_params | {'objective': 'binary:logistic', 'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1}
model = XGBClassifier(**best)
model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
proba = model.predict_proba(x_test)[:, 1]
pred = (proba >= 0.5).astype(int)
metrics = {
    'auc': float(roc_auc_score(y_test, proba)),
    'pr_auc': float(average_precision_score(y_test, proba)),
    'f1': float(f1_score(y_test, pred)),
    'brier': float(brier_score_loss(y_test, proba)),
}

mlflow.set_experiment('falabella_hybrid_ensemble')
with mlflow.start_run(run_name='hybrid_optuna_best'):
    mlflow.log_params(best)
    for k, v in metrics.items():
        mlflow.log_metric(k, float(v))
    Path('models').mkdir(parents=True, exist_ok=True)
    joblib.dump(model, 'models/hybrid_ensemble.pkl')
    mlflow.log_artifact('models/hybrid_ensemble.pkl')

metrics